# Credit Card Fraud Detection with CRISP-DM

**Goal:** Identify fraudulent transactions while balancing missed fraud and false alerts. Run all cells in order. The notebook reports measured results only after execution.

## 1. Business understanding

Research questions: (1) How rare is fraud? (2) How do transaction amounts and time vary by class? (3) Which model provides better fraud recall and precision? (4) How does a held-out transaction score? This is an educational classification exercise, not a production fraud decision system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay)

RANDOM_STATE = 42
FIG_DIR = Path('../images')
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path('../data/creditcard.csv')

## 2. Data understanding

Download the original dataset from the Kaggle source in the README, then place `creditcard.csv` in `data/`. The PCA-transformed variables V1–V28 are anonymized; they do not support interpreting specific merchant, geographic, or behavioral causes.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Download creditcard.csv from the dataset link in README.md and save to data/creditcard.csv')
df = pd.read_csv(DATA_PATH)
assert {'Time','Amount','Class'}.issubset(df.columns)
assert set(df['Class'].dropna().unique()).issubset({0,1})
print('Shape:', df.shape)
print('Fraud counts:\n', df['Class'].value_counts(dropna=False))
print('Missing values:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
display(df.describe().T)
display(df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x='Class', ax=ax)
ax.set_xticklabels(['Legitimate','Fraud'])
ax.set_title('Transaction class distribution')
fig.tight_layout(); fig.savefig(FIG_DIR/'fraud_distribution.png', dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(data=df, x='Amount', hue='Class', bins=60, log_scale=(False,True), element='step', stat='density', common_norm=False, ax=ax)
ax.set_title('Transaction amount distribution (log-scaled frequency)')
fig.tight_layout(); fig.savefig(FIG_DIR/'transaction_amount_distribution.png', dpi=160); plt.show()

display(df.groupby('Class')['Amount'].agg(['count','median','mean','max']))
print('Fraud prevalence:', f"{df['Class'].mean():.4%}")

In [ ]:
fig, ax = plt.subplots(figsize=(11,8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature correlation heatmap')
fig.tight_layout(); fig.savefig(FIG_DIR/'correlation_heatmap.png', dpi=160); plt.show()

## 3. Data preparation

Drop exact duplicates and missing records, documenting their counts. Split **before fitting transformations** to prevent leakage. Stratification maintains fraud prevalence. Class weights address imbalance without generating synthetic transactions. Anonymized PCA features cannot be translated into business explanations.

In [ ]:
before = len(df)
df = df.drop_duplicates().dropna().copy()
print('Removed records:', before-len(df))
X = df.drop(columns='Class')
y = df['Class'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Training fraud rate:', y_train.mean(), 'Test fraud rate:', y_test.mean())

## 4. Modeling

Compare class-weighted Logistic Regression (scaled) and Random Forest (class-weighted). This is a fixed, transparent baseline comparison; tuning and threshold optimization would require a separate validation split or cross-validation.

In [ ]:
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))]),
    'Random Forest': RandomForestClassifier(n_estimators=60, min_samples_leaf=2, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
}
results=[]; fitted={}; scores={}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]
    fitted[name]=model; scores[name]=prob
    results.append({'Model':name, 'Accuracy':accuracy_score(y_test,pred), 'Precision':precision_score(y_test,pred,zero_division=0), 'Recall':recall_score(y_test,pred,zero_division=0), 'F1':f1_score(y_test,pred,zero_division=0), 'PR_AUC_AP':average_precision_score(y_test,prob), 'ROC_AUC':roc_auc_score(y_test,prob)})
results_df=pd.DataFrame(results).set_index('Model')
display(results_df.round(4))

## 5. Evaluation

**Recall:** share of actual fraud identified. **Precision:** share of flagged transactions that are fraudulent. **F1:** balances precision and recall. **Average precision (AP):** summary of precision–recall performance across thresholds, useful for rare fraud. **Accuracy:** dominated by legitimate transactions. Test results are one split, not production guarantees.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax,(name,model) in zip(axes, fitted.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test), ax=ax, colorbar=False)
    ax.set_title(name)
fig.tight_layout();fig.savefig(FIG_DIR/'confusion_matrices.png',dpi=160);plt.show()

fig, ax=plt.subplots(figsize=(7,5))
for name,prob in scores.items():
    PrecisionRecallDisplay.from_predictions(y_test,prob,name=name,ax=ax)
ax.set_title('Precision–recall curves')
fig.tight_layout();fig.savefig(FIG_DIR/'precision_recall_curves.png',dpi=160);plt.show()

fig, ax=plt.subplots(figsize=(7,4))
results_df[['Precision','Recall','F1','PR_AUC_AP']].plot(kind='bar',ax=ax,rot=0)
ax.set_ylim(0,1);ax.set_title('Held-out model comparison')
fig.tight_layout();fig.savefig(FIG_DIR/'model_comparison.png',dpi=160);plt.show()

best_name=results_df['PR_AUC_AP'].idxmax()
print('Best model by average precision:',best_name)
print('No model is approved for operational deployment by this evaluation.')

## 6. Deployment demonstration: new prediction

For a valid input, use a **held-out transaction** as a demonstration rather than inventing PCA feature values. This sample was not used for model training. The output is a risk score from a classifier, not a calibrated probability of fraud; 0.5 is a demonstration threshold, not a bank-approved operating threshold.

In [ ]:
sample = X_test.iloc[[0]].copy()
actual = int(y_test.iloc[0])
model = fitted[best_name]
score = float(model.predict_proba(sample)[0,1])
label = int(model.predict(sample)[0])
print(f'Scenario: held-out transaction | model={best_name} | score={score:.4f} | predicted class={label} | actual class={actual}')
print('Interpretation:', 'Flag for analyst review' if label else 'Not flagged by default model threshold')
print('Limitations: anonymized features, historical data, no cost-sensitive threshold tuning or probability calibration.')

## Reproducibility and limitations

Dataset: ULB/Worldline credit card transactions, publicly distributed on Kaggle. Observations are historical and highly imbalanced. Random stratified splitting can overstate real-world generalization under temporal drift; a chronological validation study is a necessary next step. Do not deploy without cost analysis, fairness/privacy assessment, drift monitoring, calibrated scores, and human review.

## Verified reference results (original dataset, deduplicated)

The project was executed against the supplied ULB/Worldline dataset using the specified 80/20 split, class weighting, and a 60-tree forest. Values below are reproducibility checkpoints; rerun the preceding cells to recalculate them.

| Classifier | Accuracy | Precision | Recall | F1 | Average precision | TP | FP | FN |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.97531 | 0.05639 | 0.87368 | 0.10593 | 0.67192 | 83 | 1389 | 12 |
| Random Forest | 0.99951 | 0.97183 | 0.72632 | 0.83133 | 0.80811 | 69 | 2 | 26 |

The anonymized Amount unit is not specified, so do not assume dollars. No score should be treated as a calibrated probability of fraud.
